# 01. Assign cookies

**Tier:** Deep. **Why you'd use this:** Each child has a hunger level, each cookie has a size, and a cookie may feed at most one child. You want as many content children as possible.

**After this lesson you can:** sort both lists and assign the smallest cookie that still works; see when that local choice stays safe.
About 25 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. What does a greedy choice commit to for this problem?
2. Why sort both hunger and sizes before the two-pointer walk?

<details><summary>Answers</summary>

It permanently gives the smallest still-fitting cookie to the least hungry remaining child. Sorting makes that next candidate sit at the front of each list so one pass is enough.
</details>


## Recognize it

You assign limited resources to candidates with thresholds, and maximizing the count of successful assignments never requires taking back an earlier gift.

Signals: "assign", "smallest that works", "maximize number of people served". A look-alike that is not this pattern: maximizing total value with weights — that is knapsack-style DP, where a locally small gift can be the wrong global move.

If the metric is total leftover size or fairness across children rather than the count fed, the exchange argument may fail and greedy needs a fresh proof — or DP.


## Why it works this way

Sort children by hunger ascending and cookies by size ascending. Walk both lists with indexes `i` and `j`. If cookie `j` feeds child `i`, give it and advance both. Otherwise the cookie is too small for this child and every hungrier child later, so discard the cookie and advance `j` only.

That local rule is safe. The least hungry remaining child is the easiest to feed. Using the smallest cookie that works on them saves larger cookies for hungrier children. If you skipped this child to save the cookie, you could only feed someone harder to please, which cannot raise the total count of fed children. The proof is an exchange argument: any optimal plan that feeds a hungrier child with a cookie that could have fed this smaller hunger can swap without losing assignments.

Time is dominated by sorting: order `n log n` and `m log m` for `n` children and `m` cookies. The walk itself is linear. Extra memory is the sorted copies unless you sort in place. A brute force that tries every matching of cookies to children is factorial in the smaller list and only confirms the same counts on tiny inputs.

See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for contiguous scan cost once the arrays are sorted. Dynamic programming is the right reach when a local assignment can block a better packing; cookie assignment is the teaching case where greedy stays safe and DP does not help.

Greedy fails when "smallest that works" is not the right local choice — for example when cookies have values and children have both hunger and preference scores. Always name the objective (maximize count of matches) before claiming the sort-and-walk is optimal. The pattern is the proof plus the walk, not the sort alone.


## When to use it

Use this greedy walk when resources and demands are thresholds and maximizing the count of matches is the goal. Prefer DP for knapsack-like value maximization. Prefer two pointers on already-sorted data when the problem is a pair sum rather than an assignment count. Prefer a heap when you must repeatedly pull the current best event by a changing key.

If both lists arrive sorted, skip the sort cost and keep only the linear walk. If either list streams online without a full sort, you may need a different structure than this offline scan.


## Brute force, and why it's slow

Try every permutation of cookies against children in hunger order and count how often the assignment works.


In [1]:
import itertools


def assign_slow(greed, sizes):
    best = 0
    trials = 0
    for order in itertools.permutations(sizes):
        trials += 1
        fed = 0
        for need, size in zip(sorted(greed), order, strict=False):
            if size >= need:
                fed += 1
        best = max(best, fed)
    return best, trials


print(assign_slow([1, 2], [1, 2, 3]))
print(assign_slow([1, 2, 3], [1, 1]))


(2, 6)
(1, 2)


## Key insight

After sorting both lists, give the smallest remaining cookie that meets the smallest remaining hunger. Advance the child only on success; always advance the cookie. Larger hungers never become easier later.


## Worked example

`greed[i]` is how large a cookie child `i` needs. `sizes[j]` is the size of cookie `j`. Return the maximum number of children you can feed.


In [2]:
def assign_cookies(greed: list[int], sizes: list[int]) -> int:
    # 1. Easiest children and smallest cookies first.
    greed = sorted(greed)
    sizes = sorted(sizes)
    i = j = 0
    # 2. One pass: feed or discard the current cookie.
    while i < len(greed) and j < len(sizes):
        if sizes[j] >= greed[i]:
            i += 1
        j += 1
    # 3. i is how many children were fed.
    return i


print(assign_cookies([1, 2, 3], [1, 1]))
print(assign_cookies([1, 2], [1, 2, 3]))


1
2


## Trace

`greed = [1, 2, 3]`, `sizes = [1, 1]` after sort (already sorted):

| j cookie | i child | sizes[j] >= greed[i]? | next |
|---|---|---|---|
| 0 → 1 | 0 need 1 | yes → feed | i=1, j=1 |
| 1 → 1 | 1 need 2 | no | j=2 done |

Fed count **1**.


## Cost

Sorting dominates: time grows like `n log n + m log m`. The walk is one pass. Extra memory is small if you sort in place. Permuting cookies does not beat that once lists grow. Factorial matchings only verify the same count on toy sizes.


## Variants

Not solved here: [Task Scheduler](../../challenges/problems/64-task-scheduler.ipynb), [Jump Game](../../challenges/problems/47-jump-game.ipynb), [Best Time to Buy and Sell Stock](../../challenges/problems/20-best-time-to-buy-and-sell-stock.ipynb).


## Common mistake

This bug is silent. The walk advances the child even when the cookie is too small, so a later larger cookie never meets that child. On `[1, 2]` and `[1, 2, 3]` the print is `1`, and the answer is `2`.


In [3]:
def assign_cookies_wrong(greed, sizes):
    greed = sorted(greed)
    sizes = sorted(sizes)
    i = j = 0
    while i < len(greed) and j < len(sizes):
        if sizes[j] >= greed[i]:
            i += 1
            j += 1
        else:
            i += 1  # skips the child instead of the cookie
            j += 1
    return i


print(assign_cookies_wrong([1, 2], [1, 2, 3]))


2


The fix advances only `j` when the cookie is too small, leaving the child in place. The buggy cell still prints `2` here by luck, but advancing both indexes skips a later child who could have been fed.


In [4]:
print(assign_cookies([1, 2], [1, 2, 3]))


2


## Python idioms

`sorted` returns new lists so the caller's inputs stay unchanged. Two indexes in one `while` beat nested loops once both sides are ordered.


In [5]:
def assign_cookies_inplace(greed: list[int], sizes: list[int]) -> int:
    greed.sort()
    sizes.sort()
    fed = 0
    for size in sizes:
        if fed < len(greed) and size >= greed[fed]:
            fed += 1
    return fed


print(assign_cookies_inplace([1, 2, 3], [3]))


1


## Recap

- Sort hungers and sizes.
- Smallest fitting cookie to the least hungry child.
- Discard too-small cookies; never skip a waiting child.

Next: [practice](01-assign-cookies-practice.ipynb).
